# 01. Загрузка и индексация PHB в Qdrant

Этот ноутбук:
1. Читает PDF книги игрока D&D (PHB, русский перевод)
2. Разбивает текст на чанки с метаданными
3. Загружает чанки в локальную векторную БД Qdrant

Эмбеддинги: GigaChat Embeddings
Хранилище: Qdrant (локально на диске)

In [1]:
# Проверка окружения и загрузка переменных
import os
from pathlib import Path
from dotenv import load_dotenv

# Загружаем .env из текущей папки
load_dotenv(Path(".env"))

GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")
GIGACHAT_SCOPE = os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS")
GIGACHAT_MODEL = os.getenv("GIGACHAT_MODEL", "GigaChat-2-Max")
QDRANT_PATH = os.getenv("QDRANT_PATH", "./qdrant_storage")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "dnd_phb")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "500"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "50"))

print("GIGACHAT_CREDENTIALS:", "ОК: задан" if GIGACHAT_CREDENTIALS else "НЕ ОК: не найден")
print("QDRANT_PATH:", QDRANT_PATH)
print("QDRANT_COLLECTION:", QDRANT_COLLECTION)
print("CHUNK_SIZE:", CHUNK_SIZE)
print("CHUNK_OVERLAP:", CHUNK_OVERLAP)

GIGACHAT_CREDENTIALS: ОК: задан
QDRANT_PATH: ./qdrant_storage
QDRANT_COLLECTION: dnd_phb
CHUNK_SIZE: 500
CHUNK_OVERLAP: 50


## Шаг 1. Читаем PDF

Используем pypdf для извлечения текста постранично.

Каждая страница сохраняется с номером — это нужно для метаданных чанков.

In [2]:
from pypdf import PdfReader
from tqdm import tqdm

PDF_PATH = Path("docs/phb_ru.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF не найден: {PDF_PATH}")

reader = PdfReader(str(PDF_PATH))
total_pages = len(reader.pages)
print(f"Страниц в PDF: {total_pages}")

# Читаем все страницы
pages = []
for i, page in enumerate(tqdm(reader.pages, desc="Читаем страницы")):
    text = page.extract_text()
    if text and text.strip():  # пропускаем пустые страницы
        pages.append({
            "page": i + 1,
            "text": text.strip()
        })

print(f"\nСтраниц с текстом: {len(pages)}")
print(f"\nПример — страница {pages[0]['page']}:")
print(pages[0]["text"][:300])

Страниц в PDF: 331


Читаем страницы: 100%|██████████| 331/331 [00:06<00:00, 51.02it/s]


Страниц с текстом: 316

Пример — страница 1:
Все, что необходимо для создания персонажа
 
в
 
одной из лучших
 
в мире
 
настольных ролевых игр


## Шаг 2. Разбиваем текст на чанки

Каждая страница делится на фрагменты по ~500 символов с перекрытием 50 символов.
Перекрытие нужно чтобы смысл на границах чанков не терялся.

Для каждого чанка сохраняем метаданные:
- document_id — имя файла
- chunk_id — уникальный идентификатор вида doc_page_chunk
- source — путь к файлу
- page — номер страницы

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],
)

document_id = PDF_PATH.stem  # "phb_ru"

chunks = []
for page_data in tqdm(pages, desc="Чанкинг страниц"):
    page_num = page_data["page"]
    page_text = page_data["text"]
    
    # Разбиваем текст страницы на чанки
    splits = splitter.split_text(page_text)
    
    for chunk_idx, chunk_text in enumerate(splits):
        chunk_id = f"{document_id}_p{page_num:03d}_c{chunk_idx:02d}"
        chunks.append({
            "chunk_id": chunk_id,
            "document_id": document_id,
            "source": str(PDF_PATH),
            "page": page_num,
            "text": chunk_text,
        })

print(f"Всего чанков: {len(chunks)}")
print(f"\nПример чанка:")
print(f"  chunk_id : {chunks[10]['chunk_id']}")
print(f"  document_id: {chunks[10]['document_id']}")
print(f"  page     : {chunks[10]['page']}")
print(f"  source   : {chunks[10]['source']}")
print(f"  text     : {chunks[10]['text'][:200]}")

Чанкинг страниц: 100%|██████████| 316/316 [00:00<00:00, 11632.95it/s]

Всего чанков: 3182

Пример чанка:
  chunk_id : phb_ru_p003_c08
  document_id: phb_ru
  page     : 3
  source   : docs/phb_ru.pdf
  text     : 620А9217000001 EN 
ISBN: 987-0-7869-6560-1 
Впервые отпечатано: Август 2014 
 
9 8 7 6 5 4 3 2 1 
 
DUNGEONS & DRAGONS, D&D, WIZARDS OF THE COAST, Забытые Королевства, амперсанд в виде дракона, Книга 


## Шаг 3. Загружаем в Qdrant

Qdrant — векторная БД. Она хранит чанки вместе с эмбеддингами (числовыми представлениями текста).
По эмбеддингам потом ищем похожие фрагменты по запросу.

Эмбеддинги считаем через GigaChat Embeddings.
Qdrant запускается локально — данные хранятся в папке qdrant_storage/.

In [4]:
from langchain_gigachat import GigaChatEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Инициализируем эмбеддинги GigaChat
embeddings = GigaChatEmbeddings(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    verify_ssl_certs=False,
)

print("Проверяем эмбеддинги на одном тексте...")
test_vec = embeddings.embed_query("тестовый запрос")
print(f"ОК: Эмбеддинги работают, размерность вектора: {len(test_vec)}")

Проверяем эмбеддинги на одном тексте...
ОК: Эмбеддинги работают, размерность вектора: 1024


## Шаг 4. Создаём коллекцию и загружаем чанки

Загрузка идёт батчами по 50 чанков — чтобы не перегружать API GigaChat запросами.
Все метаданные (chunk_id, document_id, page, source) сохраняются вместе с вектором.

In [5]:
# Создаём локальный клиент Qdrant
client = QdrantClient(path=QDRANT_PATH)

# Пересоздаём коллекцию (если запускать повторно — старые данные сотрутся)
if client.collection_exists(QDRANT_COLLECTION):
    client.delete_collection(QDRANT_COLLECTION)
    print(f"Старая коллекция '{QDRANT_COLLECTION}' удалена")

client.create_collection(
    collection_name=QDRANT_COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)
print(f"Коллекция '{QDRANT_COLLECTION}' создана")

Старая коллекция 'dnd_phb' удалена
Коллекция 'dnd_phb' создана


In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document

# Превращаем чанки в Document-объекты для LangChain
documents = []
for chunk in chunks:
    doc = Document(
        page_content=chunk["text"],
        metadata={
            "chunk_id":    chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "source":      chunk["source"],
            "page":        chunk["page"],
        }
    )
    documents.append(doc)

print(f"Документов подготовлено: {len(documents)}")

# Создаём vector store через уже открытый client (не открываем второй раз)
vector_store = QdrantVectorStore(
    client=client,
    collection_name=QDRANT_COLLECTION,
    embedding=embeddings,
)

# Загружаем батчами по 50 штук
BATCH_SIZE = 50
total_batches = (len(documents) + BATCH_SIZE - 1) // BATCH_SIZE

for i in tqdm(range(0, len(documents), BATCH_SIZE), desc="Загрузка в Qdrant", total=total_batches):
    batch = documents[i : i + BATCH_SIZE]
    vector_store.add_documents(batch)

print(f"\nВсе чанки загружены в Qdrant!")

Документов подготовлено: 3182


Загрузка в Qdrant: 100%|██████████| 64/64 [00:19<00:00,  3.35it/s]


Все чанки загружены в Qdrant!


## Шаг 5. Проверяем индекс

Делаем тестовый поиск — убедимся, что чанки лежат в Qdrant и метаданные возвращаются вместе с результатами.

In [7]:
# Проверочный поиск
test_query = "какой урон наносит огненный шар"
results = vector_store.similarity_search_with_score(test_query, k=3)

print(f"Запрос: '{test_query}'")
print(f"Найдено результатов: {len(results)}\n")

for i, (doc, score) in enumerate(results, 1):
    print(f"--- Результат {i} ---")
    print(f"  chunk_id   : {doc.metadata['chunk_id']}")
    print(f"  document_id: {doc.metadata['document_id']}")
    print(f"  page       : {doc.metadata['page']}")
    print(f"  source     : {doc.metadata['source']}")
    print(f"  score      : {score:.4f}")
    print(f"  text       : {doc.page_content[:200]}")
    print()

Запрос: 'какой урон наносит огненный шар'
Найдено результатов: 3

--- Результат 1 ---
  chunk_id   : phb_ru_p197_c12
  document_id: phb_ru
  page       : 197
  source     : docs/phb_ru.pdf
  score      : 0.8356
  text       : урон огнём. 
Психическая энергия. Атаки силой разума, та-
кие как у иллитидов, причиняют урон психиче-
ской энергией. 
Рубящий. Мечи, топоры и когти чудовищ при-
чиняют рубящий урон. 
Силовое поле. Си

--- Результат 2 ---
  chunk_id   : phb_ru_p197_c08
  document_id: phb_ru
  page       : 197
  source     : docs/phb_ru.pdf
  score      : 0.8312
  text       : вает огненный шар или жрец накладывает небес-
ный огонь, урон от их заклинаний определяется 
лишь один раз и причиняется всем существам, по 
которым попал взрыв. 
 
КРИТИЧЕСКИЕ ПОПАДАНИЯ 
Если вы сове

--- Результат 3 ---
  chunk_id   : phb_ru_p270_c07
  document_id: phb_ru
  page       : 270
  source     : docs/phb_ru.pdf
  score      : 0.8295
  text       : ство, это существо должно совершить спасбросок 
от

In [8]:
test_queries = [
    "требования для мультиклассирования паладина",
    "как работает вдохновение барда",
    "что такое спасбросок",
]

for query in test_queries:
    print(f"{'='*60}")
    print(f"Запрос: '{query}'")
    results = vector_store.similarity_search_with_score(query, k=2)
    for i, (doc, score) in enumerate(results, 1):
        print(f"  [{i}] chunk_id={doc.metadata['chunk_id']} page={doc.metadata['page']} score={score:.4f}")
        print(f"       {doc.page_content[:150]}")
    print()

Запрос: 'требования для мультиклассирования паладина'
  [1] chunk_id=phb_ru_p084_c06 page=84 score=0.8446
       ние могут находиться в гармонии, или ваша 
клятва может представлять стандарты поведения, 
которых вы ещё не достигли. 
 
БЫСТРОЕ СОЗДАНИЕ 
Вы можете 
  [2] chunk_id=phb_ru_p326_c23 page=326 score=0.8419
       умения Использование заклинаний; и 
мультиклассирование

Запрос: 'как работает вдохновение барда'
  [1] chunk_id=phb_ru_p054_c04 page=54 score=0.8508
       позволяет это. 
 
ФОКУСИРОВКА ЗАКЛИНАНИЯ 
Вы можете использовать ваш музыкальный ин-
струмент (смотрите в главе 5) в качестве фокуси-
ровки для ваших 
  [2] chunk_id=phb_ru_p054_c06 page=54 score=0.8465
       Вы можете использовать это умение количе-
ство раз, равное модификатору вашей Харизмы, 
но как минимум один раз. Потраченные использо-
вания этого уме

Запрос: 'что такое спасбросок'
  [1] chunk_id=phb_ru_p273_c12 page=273 score=0.8440
       должно совершить спасбросок Ловкости. При про-
вале оно получает у

In [9]:
collection_info = client.get_collection(QDRANT_COLLECTION)
points_count = client.count(QDRANT_COLLECTION).count

print(f"Название    : {QDRANT_COLLECTION}")
print(f"Векторов    : {points_count}")
print(f"Путь к БД   : {QDRANT_PATH}")
print("\nИндекс готов к использованию через MCP")

Название    : dnd_phb
Векторов    : 3182
Путь к БД   : ./qdrant_storage

Индекс готов к использованию через MCP


## Дополнение: вторая коллекция для MCP-сервера

mcp-server-qdrant использует собственную модель эмбеддингов fastembed.
Создаём отдельную коллекцию dnd_phb_mcp с теми же чанками,
но проиндексированную через fastembed — чтобы MCP-сервер мог искать по ней.

In [10]:
from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

# Закрываем все предыдущие соединения с Qdrant
try:
    client.close()
    print("Старый client закрыт")
except:
    pass
try:
    mcp_client.close()
    print("Старый mcp_client закрыт")
except:
    pass

MCP_COLLECTION = "dnd_phb_mcp"
FASTEMBED_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

print(f"Загружаем модель {FASTEMBED_MODEL}...")
embed_model = TextEmbedding(model_name=FASTEMBED_MODEL)

test_vec = list(embed_model.embed(["тест"]))[0]
VECTOR_SIZE = len(test_vec)
print(f"Размерность вектора: {VECTOR_SIZE}")

mcp_client = QdrantClient(path=QDRANT_PATH)

if mcp_client.collection_exists(MCP_COLLECTION):
    mcp_client.delete_collection(MCP_COLLECTION)
    print(f"Старая коллекция '{MCP_COLLECTION}' удалена")

VECTOR_NAME = "fast-paraphrase-multilingual-mpnet-base-v2"
mcp_client.create_collection(
    collection_name=MCP_COLLECTION,
    vectors_config={
        VECTOR_NAME: VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    },
)
print(f"Коллекция '{MCP_COLLECTION}' создана")

Старый client закрыт
Загружаем модель sentence-transformers/paraphrase-multilingual-mpnet-base-v2...


/var/folders/60/4tt632bs2tdfbjylw4n60xgc0000gn/T/ipykernel_37487/72535840.py:22: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  embed_model = TextEmbedding(model_name=FASTEMBED_MODEL)


Размерность вектора: 768
Старая коллекция 'dnd_phb_mcp' удалена
Коллекция 'dnd_phb_mcp' создана


In [11]:
# Загружаем чанки батчами
BATCH_SIZE = 100
texts = [c["text"] for c in chunks]
total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Индексация MCP-коллекции", total=total_batches):
    batch_chunks = chunks[i : i + BATCH_SIZE]
    batch_texts = texts[i : i + BATCH_SIZE]
    
    vectors = list(embed_model.embed(batch_texts))
    
    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector={VECTOR_NAME: vectors[j].tolist()},
            payload={
                "document": batch_chunks[j]["text"],
                "chunk_id": batch_chunks[j]["chunk_id"],
                "document_id": batch_chunks[j]["document_id"],
                "source": batch_chunks[j]["source"],
                "page": batch_chunks[j]["page"],
            }
        )
        for j in range(len(batch_chunks))
    ]
    
    mcp_client.upsert(collection_name=MCP_COLLECTION, points=points)

count = mcp_client.count(MCP_COLLECTION).count
print(f"\nMCP-коллекция готова: {count} векторов в '{MCP_COLLECTION}'")

Индексация MCP-коллекции: 100%|██████████| 32/32 [07:02<00:00, 13.20s/it]


MCP-коллекция готова: 3182 векторов в 'dnd_phb_mcp'


In [13]:
# Закрываем все соединения — освобождаем папку для MCP-сервера
try:
    client.close()
except:
    pass
try:
    mcp_client.close()
except:
    pass
print("Все соединения с Qdrant закрыты. Можно переходить к 02_search_demo.ipynb")

Все соединения с Qdrant закрыты. Можно переходить к 02_search_demo.ipynb
